In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

In [2]:
def detect_page_type(soup):
    if soup.find("table", class_="wikitable"):
        return "table"
    return "category"

In [3]:
def scrape_wikipedia_category_links(url):
    soup = get_soup(url)
    painting_links = []

    # Wikipedia category pages often store links in category groups
    content = soup.find("div", id="mw-pages")
    if not content:
        return painting_links

    for a in content.find_all("a", href=True):
        href = a["href"]
        full_url = urljoin("https://en.wikipedia.org", href)

        # skip navigation-ish links
        text = a.get_text(" ", strip=True)
        if not text or text.lower() in {"next page", "previous page"}:
            continue

        painting_links.append(full_url)

    # remove duplicates
    seen = set()
    unique_links = []
    for link in painting_links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)

    return unique_links

In [4]:
import re

def scrape_wikipedia_painting_page(painting_url):
    soup = get_soup(painting_url)

    title = None
    year = None
    image_url = None

    # title
    heading = soup.find("h1", id="firstHeading")
    if heading:
        title = heading.get_text(" ", strip=True)

    # image
    infobox = soup.find("table", class_="infobox")
    if infobox:
        img = infobox.find("img")
        if img and img.get("src"):
            image_url = urljoin("https:", img["src"])

        # try to find a year in infobox rows
        rows = infobox.find_all("tr")
        for row in rows:
            header = row.find("th")
            data = row.find("td")
            if not header or not data:
                continue

            htxt = header.get_text(" ", strip=True).lower()
            dtxt = data.get_text(" ", strip=True)

            if "year" in htxt or "date" in htxt:
                year = dtxt
                break

    # fallback: regex over page text
    if not year:
        text = soup.get_text(" ", strip=True)
        match = re.search(r"\b(1[5-9]\d{2}|20\d{2})\b", text)
        if match:
            year = match.group(1)

    return {
        "title": title,
        "year": year,
        "image_url": image_url,
        "painting_url": painting_url
    }

In [5]:
def scrape_wikipedia_artist_page(url, max_items=None):
    soup = get_soup(url)
    page_type = detect_page_type(soup)

    if page_type == "table":
        return scrape_wikipedia_paintings_table(url)

    # otherwise category/list page
    links = scrape_wikipedia_category_links(url)
    if max_items is not None:
        links = links[:max_items]

    paintings = []
    for link in links:
        try:
            paintings.append(scrape_wikipedia_painting_page(link))
            print(f"Scraped: {link}")
        except Exception as e:
            print(f"Failed: {link} -> {e}")

    return paintings

In [41]:
# Test 1 with Andy Warhol, hyperlink structure, page does not directly provide the images

warhol_url = "https://en.wikipedia.org/wiki/Category:Paintings_by_Andy_Warhol"
paintings = scrape_wikipedia_artist_page(warhol_url, max_items=10)

for p in paintings:
    print(p)

Scraped: https://en.wikipedia.org/wiki/Wikipedia:FAQ/Categorization#Why_might_a_category_list_not_be_up_to_date?
Scraped: https://en.wikipedia.org/wiki/3_Coke_Bottles
Scraped: https://en.wikipedia.org/wiki/129_Die_in_Jet!
Scraped: https://en.wikipedia.org/wiki/Athletes_(Warhol_series)
Scraped: https://en.wikipedia.org/wiki/Big_Electric_Chair
Scraped: https://en.wikipedia.org/wiki/Camouflage_Self-Portrait
Scraped: https://en.wikipedia.org/wiki/Campbell%27s_Soup_Cans
Scraped: https://en.wikipedia.org/wiki/Campbell%27s_Soup_Cans_II
Scraped: https://en.wikipedia.org/wiki/Campbell%27s_Soup_I
Scraped: https://en.wikipedia.org/wiki/Cars_(painting)
{'title': 'Wikipedia : FAQ/Categorization', 'year': '2004', 'image_url': None, 'painting_url': 'https://en.wikipedia.org/wiki/Wikipedia:FAQ/Categorization#Why_might_a_category_list_not_be_up_to_date?'}
{'title': '3 Coke Bottles', 'year': '1962', 'image_url': None, 'painting_url': 'https://en.wikipedia.org/wiki/3_Coke_Bottles'}
{'title': '129 Die in 

In [7]:
# Test 2 with Claude Monet, table structure, page directly provides the paintings

monet_url = "https://en.wikipedia.org/wiki/List_of_paintings_by_Claude_Monet#"
paintings = scrape_wikipedia_artist_page(monet_url, max_items=10)

for p in paintings:
    print(p)

NameError: name 'scrape_wikipedia_paintings_table' is not defined